In [ ]:
import torch
from transformers import AutoImageProcessor, Swin2SRForImageClassification, TrainingArguments, Trainer
from PIL import Image
#test
Choose a high-res starting point
Swinv2 is superior for high-res because it uses log-spaced window scaling
model_id = "microsoft/swinv2-tiny-patch4-window8-256" 
target_resolution = 512  # You can try 768 or 1024 if your VRAM allows

Configure the Image Processor (The "Eyes")
We force it to use our high-res target instead of the default 256
processor = AutoImageProcessor.from_pretrained(model_id)
processor.size = {"height": target_resolution, "width": target_resolution}

Load Model with Position Interpolation
model = Swin2SRForImageClassification.from_pretrained(
    model_id,
    num_labels=2,
    ignore_mismatched_sizes=True,
    # This allows the model to adapt its internal grid to the new resolution
    interpolate_pos_encoding=True 
)

Move to AMD GPU (ROCm)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

Preprocessing Function
def transform(example_batch):
    # Take high-res images and convert them to the target pixel values
    inputs = processor([x for x in example_batch['image']], return_tensors='pt')
    inputs['labels'] = example_batch['label']
    return inputs

Training Arguments optimized for 32GB RAM / AMD
training_args = TrainingArguments(
    output_dir="./high_res_checkpoints",
    per_device_train_batch_size=4, # Keep low for high-res
    gradient_accumulation_steps=4, # Effectively batch size 16
    learning_rate=3e-5,
    num_train_epochs=5,
    fp16=True, # Critical for AMD GPU speed (ROCm)
    logging_steps=10,
    remove_unused_columns=False,
    push_to_hub=False,
)

Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=your_dataset["train"].with_transform(transform),
    eval_dataset=your_dataset["test"].with_transform(transform),
)

trainer.train()